# NAC COCO Instance Preprocessing

Create split `chips/` and `labels/` folders from the NAC COCO release. The notebook uses the same functions as the sbatchable Python script.

## Imports

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "nac_coco_instance_preprocessing.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()

LFM_ROOT = NOTEBOOK_DIR.parents[1]
if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

from scripts.python.instance_seg.preprocess_nac_coco_instance_dataset import (
    process_split,
    write_summary,
)

print("LFM_ROOT:", LFM_ROOT)

## Config

In [ ]:
NAC_ROOT = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final")
OUTPUT_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/256_256_inputs/nac_coco_inst_seg")

SPLITS = ["train", "val", "test"]
INCLUDE_DTM = False
NC_VARIABLE = "band_data"
SCALE_MODE = "per-chip-minmax"  # "per-chip-minmax" or "none"
CHIP_SUFFIX = "_input_nac_chip"
LABEL_SUFFIX = "_label"
MAX_WORKERS = 10
LIMIT = None
OVERWRITE = False

print("Input:", NAC_ROOT)
print("Output:", OUTPUT_ROOT)

## Inspect Input

In [ ]:
for path in sorted(NAC_ROOT.iterdir()):
    print(path.name, "dir" if path.is_dir() else "file")

for split in SPLITS:
    split_json = NAC_ROOT / "splits" / f"{split}.json"
    print(split, split_json.exists(), split_json)

## Process Splits

In [ ]:
class Args:
    nac_root = NAC_ROOT
    output_root = OUTPUT_ROOT
    include_dtm = INCLUDE_DTM
    nc_variable = NC_VARIABLE
    scale_mode = SCALE_MODE
    chip_suffix = CHIP_SUFFIX
    label_suffix = LABEL_SUFFIX

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summaries = []
for split in SPLITS:
    summary = process_split(
        nac_root=NAC_ROOT,
        output_root=OUTPUT_ROOT,
        split=split,
        include_dtm=INCLUDE_DTM,
        nc_variable=NC_VARIABLE,
        scale_mode=SCALE_MODE,
        chip_suffix=CHIP_SUFFIX,
        label_suffix=LABEL_SUFFIX,
        max_workers=MAX_WORKERS,
        limit=LIMIT,
        overwrite=OVERWRITE,
    )
    summaries.append(summary)
    print(summary)

write_summary(OUTPUT_ROOT, summaries, Args)
print("Saved preprocessed NAC dataset to:", OUTPUT_ROOT)

## Training Args

In [ ]:
print("Use these data matching args with the instance workflows:")
print("--image-glob '*.npy' --image-suffix '_input_nac_chip' --label-suffix '_label'")
print("Data root:", OUTPUT_ROOT)